# v044_stage2_m3_full — v042 + interactions, missing flags and zero-gain pruning, on the mock

| Field | Value |
|---|---|
| **Version** | `v044_stage2_m3_full` |
| **Plan group** | C5 (07 §3 interactions, 07 §1 missing flags, 08 §8 pruning) |
| **Parent version** | `v042` (`v042_stage2_m3`, est_public 0.9679); same-machine baseline: arm B below |
| **Author** | M3 |
| **Date** | 2026-09-26 |
| **Status** | investigate (not kept) |
| **Hypothesis** | The last planned M3 features (interaction flags, missing-number flags) added to v042's stage 2 raise est_public on the mock by ≥ +0.001 over v042's configuration; leaving the zero-gain columns out of stage 2 costs nothing. |

v042 put M3's four feature groups into stage 2 of v104's two-stage matcher (+0.0020
est_public). This version adds the two groups M3's plan still had open, computed like v042's on
the kept pairs (`TwoStageConfig.extra_groups`), and tests the zero-gain pruning of 08 §8
(`TwoStageConfig.drop_columns` with `features.ZERO_GAIN_COLUMNS`). One stage-1 pass (v101)
serves four stage-2 arms:

```
v101 (stage 1) -> filter -> competition + anchors + M3 groups + new groups -> stage 2 arms:
   A  v104's columns                                   reproduces v107 (est_public 0.9659)
   B  + M3's four groups                               reproduces v042 (0.9679): the baseline
   D  + interactions + missing_flags                   this version
   E  D without features.ZERO_GAIN_COLUMNS             the pruning of 08 §8
-> v107's rule tuning (tight threshold vs tight expected-F0.5) -> est_public on mock val
```

## 1. Hypothesis

* **Change vs parent:** stage 2 also reads `interactions` (`name_strong_addr_weak`,
  `addr_strong_name_weak`, `both_strong`) and `missing_flags` (`nums_empty_l`, `nums_empty_r`);
  arm E additionally leaves `sim_addr_char`, `addr_empty_r`, `addr_empty_l` and
  `postcode_prefix_eq` out of stage 2. Everything else is v042's.
* **Why it could help:** the interaction flags state the decoy (same name, other address) and
  rename (same address, other name) signatures that stage 2 otherwise assembles from two
  splits; the number flags say which side lacks house numbers when the numeric scores are NaN.
* **Prediction (falsifiable):** est_public(D) ≥ est_public(B) + 0.001; est_public(E) within
  ±0.0005 of D.
* **Expectation to beat:** trees learn such interactions themselves and stage 2 lives on the
  competition features (95 % of its gain in v043), so a null result is likely; the run settles
  M3's last open plan items with a number.
* **Discard if:** est_public(D) ≤ est_public(B).

## 2. Setup

Imports first; the cell after holds every version-specific value.

In [1]:
import json
import shutil
import subprocess
import sys
import time
from dataclasses import asdict, replace

import pandas as pd
from IPython.display import display

from entity_resolution import config as C
from entity_resolution.data import isin
from entity_resolution.decision import apply_rule, tune_expected
from entity_resolution.evaluate import blocking_report, error_samples
from entity_resolution.features import (
    DEFAULT_GROUPS, ZERO_GAIN_COLUMNS, feature_names,
)
from entity_resolution.mock import FP_WEIGHT, build_mock, target_shape
from entity_resolution.model import MatcherParams
from entity_resolution.pipeline import (
    Fitted, PipelineConfig, mem_guard, mock_scores, peak_rss_gb, tune_mock,
)
from entity_resolution.split import load_fold
from entity_resolution.tracking import log_result, timed
from entity_resolution.trainset import inner_split, sample_s1
from entity_resolution.twostage import (
    TwoStage, TwoStageConfig, fit_stage2, mock_scored, mock_stage1, run_test_two_stage,
)

pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 50)
pd.set_option("display.max_columns", 30)

### Parameters of this version

Stage 1 is v101 (`experiments/v101_name_frequency/artifacts/`, re-fitted deterministically on
this machine on day 2); the stage-2 settings are v104's plus the extra groups.

In [2]:
VERSION = "v044_stage2_m3_full"
PARENT = "v042_stage2_m3"           # logged parent (est_public 0.9679)
GROUP = "C5"
OWNER = "M3"
M3_GROUPS = ("idf", "token_freq", "ctx_idf", "address_extra")    # v042's groups
NEW_GROUPS = ("interactions", "missing_flags")                     # this version's
RUN_TEST = False                    # test inference only for shortlisted versions (§9)

cfg = PipelineConfig(feature_groups=(*DEFAULT_GROUPS, "frequency"),
                     model=MatcherParams(n_estimators=4000))          # v101 = stage 1

Derived paths, stage 1, the stage-2 configuration and the parent's logged scores. The stage-1
cache directory is specific to this combination of extra groups (`twostage._cached_stage1`
refuses a cache written under another configuration).

In [3]:
EXP_DIR = C.EXPERIMENTS / VERSION
ARTIFACTS = EXP_DIR / "artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
stage1 = Fitted.load(C.EXPERIMENTS / "v101_name_frequency" / "artifacts", cfg)
rec = json.loads((C.EXPERIMENTS / "v104_two_stage" / "metrics.json").read_text())["metrics"][
    "two_stage"]
tcfg = TwoStageConfig(**{**rec, "model": MatcherParams(**rec["model"]),
                         "train_roles": tuple(rec.get("train_roles", ("fit",))),
                         "extra_groups": (*M3_GROUPS, *NEW_GROUPS)})
STAGE1_CACHE = (cfg.cache_dir / "stage1" / f"v101_{cfg.blocking.key()}_f{tcfg.floor}"
                f"_k{tcfg.max_cands}_a{int(tcfg.anchors)}_m3x")
parent = json.loads((C.EXPERIMENTS / PARENT / "metrics.json").read_text())["metrics"]
SCORE_COLS = ["f_beta", "f_tight", "est_public", "f_beta_singletons", "f_beta_matched",
              "pair_precision", "pair_recall"]
parent_scores = {k: parent[k] for k in SCORE_COLS if k in parent}
m3_columns = feature_names(M3_GROUPS)
new_columns = feature_names(NEW_GROUPS)
timings: dict[str, float] = {}
t_start = time.time()
print(f"stage 1 rule {stage1.rule}; M3 columns {len(m3_columns)}, new columns {new_columns}")
print("zero-gain columns left out in arm E:", ZERO_GAIN_COLUMNS)
print("parent v042:", {k: round(v, 5) for k, v in parent_scores.items()})

stage 1 rule DecisionRule(tau_abs=0.42, tau_rel=0.0, tau_single=0.52, max_matches=11, one_to_one=True); M3 columns 23, new columns ['name_strong_addr_weak', 'addr_strong_name_weak', 'both_strong', 'nums_empty_l', 'nums_empty_r']
zero-gain columns left out in arm E: ('sim_addr_char', 'addr_empty_r', 'addr_empty_l', 'postcode_prefix_eq')
parent v042: {'f_beta': 0.97623, 'f_tight': 0.97508, 'est_public': 0.96788, 'f_beta_singletons': 0.98655, 'f_beta_matched': 0.97562, 'pair_precision': 0.99726, 'pair_recall': 0.93802}


## 3. Data: the mock fold

Built exactly as in v042 / v104 / v107.

In [4]:
with timed("load", timings):
    train = load_fold("train", columns=[C.COUNTRY])
    val = load_fold("val", columns=[C.COUNTRY])
    fit_fold, tune_fold = inner_split(train)
    fit_sample = sample_s1(fit_fold.s1, cfg.n_fit_s1)[C.ENTITY_ID]
    mock = build_mock(train, val, tune_fold.s1[C.ENTITY_ID], fit_sample, target_shape())
del train, val, fit_fold, tune_fold, fit_sample
display(mock.info)
mock.fold.summary()

,country,s1_train,pool_train,keep_frac,s1_kept,pool_kept,s1_present,pool_per_s1,test_pool_per_s1,present_val,present_tune,present_fit
0,India,883188,4133346,1.000,883188,4133346,709678,5.824,5.824,176522,176208,356948
1,US,1323633,6186873,0.617,815850,3816702,663049,5.756,5.756,163562,163325,336162


{'fold': 'mock',
 's1': 1372727,
 's2': 3878856,
 's3': 4071192,
 'true_pairs': 4753992,
 'singleton_share': 0.0558}

## 4. Method

### 4.1 Stage 1, the filter and every extra group (as v042, plus the new groups)

The mock's blocking is cached from v042, so this pass re-scores stage 1 and builds the six
extra groups on the kept pairs.

In [5]:
t0 = time.time()
outs = mock_stage1(cfg, stage1, mock, tcfg, timings=timings, cache_dir=STAGE1_CACHE / "mock")
timings["stage1_seconds"] = round(time.time() - t0, 2)
kept = pd.concat([o.pairs for o in outs.values()], ignore_index=True)
print(f"stage 1 {timings['stage1_seconds']:.0f} s; pairs {sum(o.n_all for o in outs.values()):,}"
      f" -> kept {len(kept):,}; frame columns {next(iter(outs.values())).X.shape[1]}")
filter_report = pd.DataFrame({r: blocking_report(kept, mock.part(r))
                              for r in ("fit", "tune", "val")}).T
del kept
filter_report[["pair_recall", "entity_recall", "ceiling_f_beta", "candidates_mean",
               "candidates_p95"]]

stage 1 2115 s; pairs 47,348,383 -> kept 6,311,331; frame columns 98


,pair_recall,entity_recall,ceiling_f_beta,candidates_mean,candidates_p95
fit,0.964196,0.996389,0.987477,4.601084,9.0
tune,0.964089,0.996298,0.987441,4.595480,9.0
val,0.964386,0.996392,0.987585,4.592856,9.0


### 4.2 Four stage-2 arms and v107's rule tuning

Each arm trains v104's cross-fitted XGBoost stage 2 on its columns (arm E through
`drop_columns`), scores the mock's tune and val entities, tunes v107's two rule families for the
tight score and keeps the better, then scores the val entities.

In [6]:
all_cols = list(next(iter(outs.values())).X.columns)
ARMS = {
    "A: v104 columns": ([c for c in all_cols if c not in m3_columns + new_columns], tcfg),
    "B: + M3 groups (v042)": ([c for c in all_cols if c not in new_columns], tcfg),
    "D: + interactions + missing_flags": (all_cols, tcfg),
    "E: D without zero-gain columns": (all_cols, replace(tcfg, drop_columns=ZERO_GAIN_COLUMNS)),
}
TUNE_KW = {"gammas": (0.7, 0.85, 1.0, 1.2, 1.5, 2.0), "misses": (0.0, 0.05, 0.1, 0.2, 0.4),
           "fp_weight": FP_WEIGHT}


def run_arm(columns: list[str], arm_cfg: TwoStageConfig) -> dict:
    """Stage 2 on ``columns`` under ``arm_cfg``, mock scores, v107's rule choice, val scores.

    Returns the models, their fit info, the scored tune + val pairs, the chosen rule with its
    tuning table, ``mock_scores`` of the val entities, the mean gain share per feature and the
    seconds taken.
    """
    t0 = time.time()
    models, info = fit_stage2(outs, mock, arm_cfg, columns=columns)
    scored, _ = mock_scored(outs, models, mock, arm_cfg)
    rule_t, table_t = tune_mock(scored, mock, cfg.grid, fp_weight=FP_WEIGHT)
    tune_part = mock.part("tune")
    rows_t = scored[isin(scored[C.S1_ID], pd.Index(tune_part.s1[C.ENTITY_ID]))]
    rule_e, table_e = tune_expected(rows_t, tune_part.s1[C.ENTITY_ID], tune_part.pairs,
                                    **TUNE_KW)
    best_t, best_e = float(table_t["f_beta"].max()), float(table_e["f_beta"].max())
    rule, table = (rule_e, table_e) if best_e > best_t else (rule_t, table_t)
    importance = pd.concat([m.importance() for m in models], axis=1).mean(axis=1)
    return {"models": models, "info": info, "scored": scored, "rule": rule, "table": table,
            "res": mock_scores(scored, mock, rule), "tune_threshold": best_t,
            "tune_expected": best_e, "seconds": round(time.time() - t0, 1),
            "n_columns": len(models[0].feature_names_),
            "importance": importance.sort_values(ascending=False)}

## 5. Evaluation on the mock

### 5.1 The arms

In [7]:
arms: dict[str, dict] = {}
for name, (cols, arm_cfg) in ARMS.items():
    arms[name] = run_arm(cols, arm_cfg)
    r = arms[name]["res"].loc["all"]
    print(f"{name:36s} {arms[name]['n_columns']:3d} cols {arms[name]['seconds']:6.0f} s  "
          f"est_public {r['est_public']:.5f}  mock F0.5 {r['f_beta']:.5f}  "
          f"rule {arms[name]['rule']}")
    mem_guard(name)
timings["arms_seconds"] = round(sum(a["seconds"] for a in arms.values()), 1)

A: v104 columns                       70 cols    413 s  est_public 0.96588  mock F0.5 0.97451  rule ExpectedRule(gamma=1.5, miss=0.05, max_matches=11, one_to_one=True)


B: + M3 groups (v042)                 93 cols    526 s  est_public 0.96788  mock F0.5 0.97623  rule ExpectedRule(gamma=1.5, miss=0.0, max_matches=11, one_to_one=True)


D: + interactions + missing_flags     98 cols    894 s  est_public 0.96810  mock F0.5 0.97647  rule ExpectedRule(gamma=1.5, miss=0.2, max_matches=11, one_to_one=True)


E: D without zero-gain columns        94 cols    549 s  est_public 0.96803  mock F0.5 0.97637  rule ExpectedRule(gamma=1.5, miss=0.0, max_matches=11, one_to_one=True)


### 5.2 Arms against each other and against the logged v042

Arm A should reproduce v107 (0.96588) and arm B v042 (0.96788); the deltas are against B.

In [8]:
BASE = "B: + M3 groups (v042)"
table = pd.DataFrame({name: a["res"].loc["all", SCORE_COLS] for name, a in arms.items()}).T
table.loc["v042 (logged)"] = pd.Series(parent_scores).reindex(SCORE_COLS)
table = table.astype(float)
table["d_est_vs_B"] = table["est_public"] - table.loc[BASE, "est_public"]
table["d_f05_vs_B"] = table["f_beta"] - table.loc[BASE, "f_beta"]
display(table.round(5))
by_country = pd.concat({name: a["res"].drop(index="all")[["f_beta", "est_public",
                                                          "f_beta_singletons"]]
                        for name, a in arms.items()})
by_country.round(5)

,f_beta,f_tight,est_public,f_beta_singletons,f_beta_matched,pair_precision,pair_recall,d_est_vs_B,d_f05_vs_B
A: v104 columns,0.97451,0.97308,0.96588,0.98367,0.97397,0.99648,0.93423,-0.00200,-0.00172
B: + M3 groups (v042),0.97623,0.97508,0.96788,0.98655,0.97562,0.99726,0.93802,0.00000,0.00000
D: + interactions + missing_flags,0.97647,0.97530,0.96810,0.98613,0.97590,0.99721,0.93874,0.00022,0.00024
E: D without zero-gain columns,0.97637,0.97523,0.96803,0.98676,0.97575,0.99728,0.93824,0.00015,0.00014
v042 (logged),0.97623,0.97508,0.96788,0.98655,0.97562,0.99726,0.93802,0.00000,0.00000


f_beta  est_public  f_beta_singletons
A: v104 columns                   India  0.97014     0.96141            0.98145
                                  US     0.97923     0.97070            0.98607
B: + M3 groups (v042)             India  0.97216     0.96377            0.98559
                                  US     0.98062     0.97232            0.98759
D: + interactions + missing_flags India  0.97243     0.96401            0.98468
                                  US     0.98083     0.97251            0.98770
E: D without zero-gain columns    India  0.97229     0.96390            0.98518
                                  US     0.98076     0.97249            0.98846

### 5.3 What stage 2 reads from the new groups (arm D)

In [9]:
imp = arms["D: + interactions + missing_flags"]["importance"]
new_imp = pd.DataFrame({"gain": imp.reindex(new_columns).fillna(0.0),
                        "rank": imp.rank(ascending=False).reindex(new_columns)})
print("share of stage-2 gain: new groups", round(float(new_imp["gain"].sum()), 5),
      "| M3 groups", round(float(imp.reindex(m3_columns).fillna(0).sum()), 5))
display(new_imp.sort_values("gain", ascending=False))
imp.head(25).rename("gain share").to_frame()

share of stage-2 gain: new groups 0.00084 | M3 groups 0.02478


,gain,rank
feature,,
nums_empty_r,0.000596,56.0
nums_empty_l,0.000121,81.0
both_strong,0.000079,83.0
addr_strong_name_weak,0.000037,88.0
name_strong_addr_weak,0.000004,91.0


,gain share
feature,
pool_gap,0.707630
p1,0.191922
s1_gap,0.012480
pool_best_other,0.005813
pool_p1_sum,0.005396
s1_p1_sum,0.004431
freq_addr_r,0.003851
idf_name_cover_r,0.002963
sim_name_addr_word,0.002237


## 6. Error analysis: arm B against arm D

Counts on the mock val entities under each arm's own rule, and the pairs the new groups fix or
break.

In [10]:
KINDS = ("false_merge", "missed", "false_singleton", "singleton_merge")
part = mock.part("val")
val_ids = pd.Index(part.s1[C.ENTITY_ID])
matches: dict[str, pd.DataFrame] = {}
counts: dict[str, dict] = {}
for name in (BASE, "D: + interactions + missing_flags", "E: D without zero-gain columns"):
    a = arms[name]
    matches[name] = apply_rule(a["scored"][isin(a["scored"][C.S1_ID], val_ids)], a["rule"])
    counts[name] = {k: len(error_samples(matches[name], part, k, n=10**9)) for k in KINDS}


def keys(df: pd.DataFrame) -> set[str]:
    """``source1_entity_id|entity_id`` of every row (sets of pairs)."""
    return set((df[C.S1_ID].astype(str) + "|" + df[C.ENTITY_ID].astype(str)).tolist())


truth = keys(part.pairs)
b_s, d_s = keys(matches[BASE]), keys(matches["D: + interactions + missing_flags"])
transitions = {
    "fixed: false merge dropped": len({k for k in b_s - d_s if k not in truth}),
    "fixed: miss now predicted": len({k for k in d_s - b_s if k in truth}),
    "broken: new false merge": len({k for k in d_s - b_s if k not in truth}),
    "broken: true pair lost": len({k for k in b_s - d_s if k in truth}),
}
display(pd.DataFrame(counts))
pd.Series(transitions, name="pairs B -> D").to_frame()

,B: + M3 groups (v042),D: + interactions + missing_flags,E: D without zero-gain columns
false_merge,2744,2788,2721
missed,69866,69070,69611
false_singleton,3063,3007,3048
singleton_merge,289,299,285


,pairs B -> D
fixed: false merge dropped,193
fixed: miss now predicted,1527
broken: new false merge,247
broken: true pair lost,675


## 7. Log the result

The version's number is arm D; arms A, B and E go into `metrics.json`. Decision against the
same-machine arm B (v042's configuration): KEEP if est_public(D) > est_public(B) + 0.001, DROP
if not above B, else INVESTIGATE. Arm D's pipeline is saved for test inference.

In [11]:
d = arms["D: + interactions + missing_flags"]
est_d = float(d["res"].loc["all", "est_public"])
est_b = float(arms[BASE]["res"].loc["all", "est_public"])
est_e = float(arms["E: D without zero-gain columns"]["res"].loc["all", "est_public"])
DECISION = "KEEP" if est_d > est_b + 0.001 else ("DROP" if est_d <= est_b else "INVESTIGATE")
ts = TwoStage(stage1, d["models"], d["rule"], tcfg, d["table"], {"fit": d["info"]})
ts.save(ARTIFACTS / "two_stage")
d["scored"].to_parquet(ARTIFACTS / "mock_scored_D.parquet", index=False)
record = {
    "hypothesis": "interaction and missing-number flags in v042's stage 2 raise est_public by "
                  ">= 0.001; leaving the zero-gain columns out of stage 2 costs nothing",
    "stage1": "v101", "two_stage": tcfg.record(), "extra_groups": list(tcfg.extra_groups),
    "zero_gain_columns": list(ZERO_GAIN_COLUMNS),
    "rule": asdict(d["rule"]), "rule_kind": type(d["rule"]).__name__,
    **{k: float(d["res"].loc["all", k]) for k in SCORE_COLS},
    **{f"mock_{c}": float(d["res"].loc[c, "f_beta"]) for c in d["res"].index if c != "all"},
    "arms": {name: {**{k: float(x["res"].loc["all", k]) for k in SCORE_COLS},
                    "rule": asdict(x["rule"]), "rule_kind": type(x["rule"]).__name__,
                    "n_columns": x["n_columns"], "seconds": x["seconds"]}
             for name, x in arms.items()},
    "parent_logged": parent_scores, "new_importance": new_imp["gain"].to_dict(),
    "errors_mock": counts, "transitions": transitions, **timings,
    "peak_rss_gb": peak_rss_gb(), "decision": DECISION,
}
row = log_result(
    EXP_DIR, change="v042 + interactions and missing_flags groups in stage 2 (kept pairs); "
                    "v107 rule tuning; pruning arm",
    group=GROUP, mock_f05=float(d["res"].loc["all", "f_beta"]), cand_recall=None,
    notes=(f"est_public {est_d:.4f} (same-machine v042 config {est_b:.4f}); without zero-gain "
           f"columns {est_e:.4f}"),
    metrics=record, owner=OWNER, parent="v042", decision=DECISION)
print(f"est_public B {est_b:.5f} -> D {est_d:.5f} (E {est_e:.5f}): {DECISION}")
row

est_public B 0.96788 -> D 0.96810 (E 0.96803): INVESTIGATE


{'version': 'v044',
 'date': '2026-09-26',
 'group': 'C5',
 'change': 'v042 + interactions and missing_flags groups in stage 2 (kept pairs); v107 rule tuning; pruning arm',
 'local_f05': '',
 'mock_f05': '0.9765',
 'cand_recall': '',
 'public_f05': '',
 'commit': '20486a4',
 'notes': 'est_public 0.9681 (same-machine v042 config 0.9679); without zero-gain columns 0.9680',
 'owner': 'M3',
 'parent': 'v042',
 'decision': 'INVESTIGATE'}

## 8. Conclusion

* **Result (§5.2):** arms A and B reproduce v107 (0.96588) and v042 (0.96788) exactly. Adding
  the interaction and missing-number flags (arm D) gives est_public **0.96810 (+0.00022)** and
  mock F0.5 0.97647 (+0.00024); leaving the zero-gain columns out as well (arm E) gives
  0.96803 (+0.00015).
* **Decision (§7): INVESTIGATE**, not KEEP: the gain is a fifth of the 0.001 margin and mixed
  (misses 69,866 → 69,070 but false merges 2,744 → 2,788, singletons 0.98655 → 0.98613).
  v042 stays M3's version for test inference and upload.
* **New groups in stage 2 (§5.3):** 0.08 % of the gain; `nums_empty_r` (#56) is the only one
  used, the three interaction flags are near zero. As 07 §3 expected, the trees already build
  these signatures from `core_token_set`, `ad_jaccard` and `ad_token_set`, and stage 2 lives on
  the competition features (`pool_gap` 71 %).
* **Pruning (arm E):** removing `sim_addr_char`, `addr_empty_r`, `addr_empty_l` and
  `postcode_prefix_eq` from stage 2 changes est_public by −0.00007 against arm D and gives the
  best singletons (0.98676): they carry no signal, so 08 §8's pruning is harmless either way.
* **Next experiment:** none from M3's plan: every planned feature is built and measured; the
  groups stay opt-in, and v042 goes to test inference.


## 9. Test inference (shortlisted versions only)

With `RUN_TEST = True`: test files from arm D's saved pipeline (stage-1 outputs of the test
partitions cached per country), copied to `submissions/v044/`, then both validators.

In [12]:
if RUN_TEST:
    t0 = time.time()
    match_path, cand_path, s1n_test, test_matches, test_summary = run_test_two_stage(
        cfg, ts, cache_dir=STAGE1_CACHE / "test")
    print(f"run_test {time.time() - t0:.0f} s")
    dest = C.ROOT / "submissions" / "v044"
    dest.mkdir(parents=True, exist_ok=True)
    for p in (match_path, cand_path):
        shutil.copy2(p, dest / p.name)
    for cmd in ([sys.executable, "-m", "entity_resolution.submission", "--output-dir",
                 str(C.OUTPUT), "--check-ids"],
                [sys.executable, str(C.OFFICIAL_VALIDATOR), "--matching", str(match_path),
                 "--candidate", str(cand_path), "--test-dir", str(C.DATASET / "test")]):
        out = subprocess.run(cmd, capture_output=True, text=True)
        print(out.stdout[-2000:], out.stderr[-2000:])
print(f"notebook total {time.time() - t_start:.0f} s, peak RSS {peak_rss_gb()} GB")

notebook total 4671 s, peak RSS 5.0 GB
